# 02 — Spatial CNN Training

Trains the custom 5-block CNN on 224×224 RGB face crops from FF++ only.

**Environment:** Colab T4 (15 GB VRAM) or A100  
**Batch size:** 64 on A100 | 32 on T4  
**Checkpointing:** every epoch → Google Drive  
**Mixed precision:** `torch.cuda.amp` (required)

In [ ]:
# ── Colab setup (run every session — VMs reset between sessions) ─────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys

!pip install -q torch torchvision opencv-python-headless scikit-learn pandas tqdm

DRIVE_ROOT = '/content/drive/MyDrive/deepfake_detection'
SPLITS_DIR = f'{DRIVE_ROOT}/splits'
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints/spatial'
os.makedirs(CKPT_DIR, exist_ok=True)

# Clone repo for model/util code (pull latest if already cloned)
if not os.path.exists('/content/repo'):
    os.system('git clone https://github.com/ayrabia/DeepFake-Detection.git /content/repo')
else:
    os.system('cd /content/repo && git pull')
sys.path.insert(0, '/content/repo')

print(f'Drive root:     {DRIVE_ROOT}')
print(f'Splits dir:     {SPLITS_DIR}')
print(f'Checkpoint dir: {CKPT_DIR}')

In [ ]:
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from models.spatial.spatial_cnn import SpatialCNN

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram_gb:.1f} GB')
    BATCH_SIZE = 64 if vram_gb > 20 else 32  # 64 on A100, 32 on T4
else:
    BATCH_SIZE = 4

EPOCHS = 20
LR     = 1e-4
CKPT_PATH = f'{CKPT_DIR}/spatial_latest.pt'

model  = SpatialCNN(dropout=0.5).to(device)
scaler = GradScaler(enabled=(device == 'cuda'))
optim  = torch.optim.Adam(model.parameters(), lr=LR)
print(f'Batch size: {BATCH_SIZE}  |  Epochs: {EPOCHS}')

In [ ]:
# ── Checkpoint helpers ────────────────────────────────────────────────────────
import os

def save_checkpoint(model, optimizer, epoch, val_auc, path):
    torch.save({
        'epoch':     epoch,
        'model':     model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'val_auc':   val_auc,
    }, path)
    print(f'  Checkpoint saved → {path}')

def load_checkpoint(model, optimizer, path):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    print(f'  Resumed from epoch {ckpt["epoch"]}  val_auc={ckpt["val_auc"]:.4f}')
    return ckpt['epoch']

CKPT_PATH = os.path.join(CKPT_DIR, 'spatial_latest.pt')
start_epoch = 0
if os.path.exists(CKPT_PATH):
    start_epoch = load_checkpoint(model, optim, CKPT_PATH)

In [ ]:
import cv2
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


class SpatialDataset(Dataset):
    def __init__(self, csv_path, split, drive_root):
        self.df = pd.read_csv(csv_path)
        self.drive_root = drive_root  # e.g. /content/drive/MyDrive/deepfake_detection

        if split == 'train':
            self.transform = T.Compose([
                T.ToPILImage(),
                T.RandomHorizontalFlip(p=0.5),
                T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                T.ToTensor(),
                T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ])
        else:
            self.transform = T.Compose([
                T.ToPILImage(),
                T.ToTensor(),
                T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ])

    def _remap(self, path):
        # CSV stores relative paths: 'data/processed/faces/...'
        # Remap to: DRIVE_ROOT/processed/faces/...
        return os.path.join(self.drive_root, path.replace('data/', '', 1))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self._remap(str(row['frame_path']))
        img = cv2.imread(img_path)
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        label = torch.tensor(float(row['label']), dtype=torch.float32)
        return self.transform(img), label


# Class imbalance: weight the loss toward the minority (real) class
train_df = pd.read_csv(f'{SPLITS_DIR}/train.csv')
n_real   = (train_df['label'] == 0).sum()
n_fake   = (train_df['label'] == 1).sum()
pos_weight = torch.tensor([n_real / n_fake], dtype=torch.float32).to(device)
print(f'Class balance — real: {n_real:,}  fake: {n_fake:,}  pos_weight: {pos_weight.item():.3f}')

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

train_dataset = SpatialDataset(f'{SPLITS_DIR}/train.csv', 'train', DRIVE_ROOT)
val_dataset   = SpatialDataset(f'{SPLITS_DIR}/val.csv',   'val',   DRIVE_ROOT)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset):,} samples  |  Val: {len(val_dataset):,} samples')
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np


def train_epoch(model, loader, optimizer, loss_fn, scaler, device):
    model.train()
    total_loss = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast(enabled=(device == 'cuda')):
            logits = model(imgs)
            loss   = loss_fn(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        with autocast(enabled=(device == 'cuda')):
            probs = torch.sigmoid(model(imgs)).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())
    auc = roc_auc_score(all_labels, all_probs)
    acc = ((np.array(all_probs) > 0.5) == np.array(all_labels)).mean()
    return auc, acc


best_auc = 0.0
for epoch in range(start_epoch, EPOCHS):
    train_loss       = train_epoch(model, train_loader, optim, loss_fn, scaler, device)
    val_auc, val_acc = evaluate(model, val_loader, device)
    print(f'Epoch {epoch+1:>2}/{EPOCHS}  loss={train_loss:.4f}  val_auc={val_auc:.4f}  val_acc={val_acc:.4f}')

    save_checkpoint(model, optim, epoch+1, val_auc, CKPT_PATH)

    if val_auc > best_auc:
        best_auc = val_auc
        save_checkpoint(model, optim, epoch+1, val_auc,
                        f'{CKPT_DIR}/spatial_best.pt')
        print(f'  ★ New best val AUC: {best_auc:.4f}')

In [ ]:
from utils.results_logger import log_results

# Load best checkpoint for final evaluation
load_checkpoint(model, optim, f'{CKPT_DIR}/spatial_best.pt')

test_dataset = SpatialDataset(f'{SPLITS_DIR}/test.csv', 'test', DRIVE_ROOT)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

test_auc, test_acc = evaluate(model, test_loader, device)
print(f'Test AUC: {test_auc:.4f}  |  Test Acc: {test_acc:.4f}')

# Per-manipulation AUC
@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    all_probs = []
    for imgs, _ in loader:
        imgs = imgs.to(device)
        with autocast(enabled=(device == 'cuda')):
            probs = torch.sigmoid(model(imgs)).cpu().numpy()
        all_probs.extend(probs)
    return np.array(all_probs)

test_df = pd.read_csv(f'{SPLITS_DIR}/test.csv').reset_index(drop=True)
test_df['prob'] = predict_all(model, test_loader, device)

per_manip = {}
print('\nPer-manipulation AUC:')
for manip, group in test_df.groupby('manipulation_type'):
    if group['label'].nunique() < 2:
        continue
    auc = roc_auc_score(group['label'], group['prob'])
    per_manip[manip] = round(auc, 4)
    print(f'  {manip:<20} AUC={auc:.4f}')

metrics = {
    'val_auc':           round(best_auc, 4),
    'test_auc':          round(test_auc, 4),
    'test_acc':          round(float(test_acc), 4),
    'per_manipulation':  per_manip,
    'epochs':            EPOCHS,
    'batch_size':        BATCH_SIZE,
}
log_results('spatial_cnn', metrics)
print('\nResults logged to results/summary.json')